# Nanograd vs PyTorch Performance Comparison
This notebook benchmarks the computation speed of your custom Nanograd engine against PyTorch on both CPU and GPU (CUDA).

In [3]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from nanograd import Tensor
from nanograd.nn import MLP, MSE
from nanograd.optim import Adam

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    CUPY_AVAILABLE = False

print(f"CuPy / CUDA Available: {CUPY_AVAILABLE}")
print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")

CuPy / CUDA Available: True
PyTorch CUDA Available: True


### Configurations and Data Setup

In [4]:
BATCH_SIZE = 128
INPUT_DIM = 1024
HIDDEN_1 = 1024
HIDDEN_2 = 512
OUTPUT_DIM = 10
NUM_ITERATIONS = 20

np.random.seed(42)
X_np = np.random.randn(BATCH_SIZE, INPUT_DIM).astype(np.float32)
Y_np = np.random.randn(BATCH_SIZE, OUTPUT_DIM).astype(np.float32)

### Nanograd Benchmark Function

In [5]:
def run_nanograd_benchmark(device='cpu'):
    x = Tensor(X_np.copy())
    y = Tensor(Y_np.copy())
    if device == 'cuda':
        x = x.cuda()
        y = y.cuda()
        
    model = MLP([INPUT_DIM, HIDDEN_1, HIDDEN_2, OUTPUT_DIM])
    if device == 'cuda':
        model.cuda()
        
    optimizer = Adam(model.parameters(), learning_rate=0.01)
    loss_fn = MSE()
    if device == 'cuda':
        loss_fn.cuda()
        
    # Warmup
    _ = loss_fn(model(x), y).backward()
    if device == 'cuda':
        cp.cuda.Device().synchronize()
        
    start_time = time.perf_counter()
    for _ in range(NUM_ITERATIONS):
        predictions = model(x)
        loss = loss_fn(predictions, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if device == 'cuda':
        cp.cuda.Device().synchronize()
        
    end_time = time.perf_counter()
    return end_time - start_time

### PyTorch Benchmark Function

In [6]:
def run_pytorch_benchmark(device='cpu'):
    device_name = 'cuda' if device == 'cuda' else 'cpu'
    x = torch.tensor(X_np.copy(), requires_grad=False).to(device_name)
    y = torch.tensor(Y_np.copy(), requires_grad=False).to(device_name)
    
    model = nn.Sequential(
        nn.Linear(INPUT_DIM, HIDDEN_1),
        nn.ReLU(),
        nn.Linear(HIDDEN_1, HIDDEN_2),
        nn.ReLU(),
        nn.Linear(HIDDEN_2, OUTPUT_DIM)
    ).to(device_name).double()
    
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    
    # Warmup
    _ = loss_fn(model(x.double()), y.double()).backward()
    if device == 'cuda':
        torch.cuda.synchronize()
        
    start_time = time.perf_counter()
    for _ in range(NUM_ITERATIONS):
        predictions = model(x.double())
        loss = loss_fn(predictions, y.double())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if device == 'cuda':
        torch.cuda.synchronize()
        
    end_time = time.perf_counter()
    return end_time - start_time

### Run Comparison Benchmark

In [7]:
print("Running CPU Benchmark...")
nano_cpu = run_nanograd_benchmark('cpu')
torch_cpu = run_pytorch_benchmark('cpu')
print(f"Nanograd CPU: {nano_cpu:.4f}s | PyTorch CPU: {torch_cpu:.4f}s")

if CUPY_AVAILABLE and torch.cuda.is_available():
    print("\nRunning GPU Benchmark...")
    nano_gpu = run_nanograd_benchmark('cuda')
    torch_gpu = run_pytorch_benchmark('cuda')
    print(f"Nanograd GPU: {nano_gpu:.4f}s | PyTorch GPU: {torch_gpu:.4f}s")
    
    print(f"\nNanograd GPU Speedup: {nano_cpu / nano_gpu:.2f}x faster than CPU")
    print(f"PyTorch GPU vs Nanograd GPU: PyTorch is {nano_gpu / torch_gpu:.2f}x faster")

Running CPU Benchmark...
Nanograd CPU: 1.0821s | PyTorch CPU: 0.3171s

Running GPU Benchmark...
Nanograd GPU: 0.2766s | PyTorch GPU: 0.1096s

Nanograd GPU Speedup: 3.91x faster than CPU
PyTorch GPU vs Nanograd GPU: PyTorch is 2.53x faster
